# **Video Processing Week 3: Analisis Tangan & Aplikasi Interaktif**

Minggu ini kita akan mempelajari **analisis tangan dan aplikasi interaktif** menggunakan teknologi AI melalui MediaPipe. Fokus utama adalah deteksi dan tracking tangan serta jari untuk membangun aplikasi interaktif yang dapat merespons gestur tangan secara real-time.

**Tujuan Pembelajaran:**

Di akhir sesi ini kita akan mampu:
- Menggunakan MediaPipe Hand untuk mendeteksi dan melacak tangan secara real-time
- Mengimplementasikan sistem pengenalan gestur sederhana berdasarkan posisi landmark
- Membangun aplikasi penghitung jari dan deteksi gestur dasar
- Mengembangkan proyek Virtual Painter menggunakan gestur tangan sebagai kontrol

**Topik Praktik:**
- **Hand Landmark Detection**: Deteksi dan tracking 21 titik landmark pada tangan
- **Hand Gesture Recognition**: Pengenalan gestur sederhana seperti menghitung jari dan membedakan kepalan vs telapak terbuka
- **Virtual Painter Project**: Aplikasi interaktif untuk menggambar menggunakan gestur jari telunjuk dan mengubah warna/menghapus dengan gestur telapak terbuka

> *This module is inspired by the development of last semester’s materials.* 


*Sebelum lanjut, kita coba import library yang akan kita butuhkan dulu*

In [ ]:
# pip install opencv-contrib-python numpy matplotlib mediapipe ipykernel
# atau
# uv pip install opencv-contrib-python numpy matplotlib mediapipe ipykernel

In [1]:
import cv2
import numpy as np
import mediapipe as mp
import matplotlib.pyplot as plt
import os

## Hand Landmark Detection

kita akan melakukan deteksi 21 titik landmark pada satu atau dua tangan melalui webcam secara real-time

In [5]:
import cv2
import os

# 1. Load detektor wajah bawaan OpenCV (Haar Cascade)
# File xml ini sudah otomatis ada saat kamu menginstal OpenCV
cascade_path = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
face_cascade = cv2.CascadeClassifier(cascade_path)

# 2. Inisialisasi Kamera (0 untuk webcam bawaan laptop)
cap = cv2.VideoCapture(0)

# 3. Inisialisasi CSRT Tracker dengan proteksi versi OpenCV
try:
    tracker = cv2.TrackerCSRT.create()
except AttributeError:
    tracker = cv2.TrackerCSRT_create()

# Variabel kontrol status tracking
is_tracking_initialized = False
bbox = None

print("Membuka kamera... Hadapkan wajah ke kamera untuk mengunci otomatis.")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        print("Error: Gagal mengambil gambar dari kamera.")
        break
        
    # Balik frame secara horizontal agar seperti cermin
    frame = cv2.flip(frame, 1)

    # FASE 1: JIKA TRACKER BELUM AKTIF -> Cari wajah menggunakan Haar Cascade
    if not is_tracking_initialized:
        # Haar Cascade membutuhkan gambar grayscale (hitam putih)
        gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        
        # Deteksi wajah (scaleFactor dan minNeighbors mengatur sensitivitas)
        faces = face_cascade.detectMultiScale(gray_frame, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))
        
        if len(faces) > 0:
            # Ambil koordinat wajah pertama yang ditemukan (x, y, w, h)
            (x, y, w, h) = faces[0]
            bbox = (x, y, w, h)
            
            # Inisialisasi CSRT Tracker dengan koordinat tersebut
            tracker.init(frame, bbox)
            is_tracking_initialized = True
            print("Wajah terdeteksi! CSRT Tracker mulai mengunci target.")

        # Tampilkan teks status saat mencari wajah
        cv2.putText(frame, "Mencari Wajah (OpenCV)...", (30, 40), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 165, 255), 2)

    # FASE 2: JIKA TRACKER SUDAH AKTIF -> Lacak gerakan menggunakan CSRT Tracker
    else:
        success, bbox = tracker.update(frame)
        
        if success:
            # Jika pelacakan berhasil, gambar kotak hijau (Bounding Box)
            x, y, w, h = [int(v) for v in bbox]
            cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
            cv2.putText(frame, "TRACKING LOCK", (x, y - 10), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        else:
            # Jika wajah keluar frame atau terhalang total
            cv2.putText(frame, "Target Lost! Mencari ulang...", (30, 40), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
            # Reset status agar mendeteksi ulang wajah dari awal
            is_tracking_initialized = False

    # Tampilkan output video secara real-time
    cv2.imshow('Face Tracking (Haar Cascade + CSRT)', frame)
    
    # Tekan 'q' atau tombol 'ESC' untuk keluar
    key = cv2.waitKey(1) & 0xFF
    if key == ord('q') or key == 27:
        break

# Bersihkan dan matikan kamera
cap.release()
cv2.destroyAllWindows()
print("Kamera ditutup, program selesai.")

Membuka kamera... Hadapkan wajah ke kamera untuk mengunci otomatis.
Wajah terdeteksi! CSRT Tracker mulai mengunci target.
Kamera ditutup, program selesai.


## Hand Gesture Recognition Sederhana

Kita akan mengimplementasikan pengenalan gestur sederhana dengan menerjemahkan data landmark menjadi informasi gestur seperti menghitung jari yang terangkat dan membedakan antara kepalan tangan dengan telapak terbuka.

### Menerjemahkan data landmark menjadi gestur

Berdasarkan kode sebelumnya, MediaPipe hanya memberi Anda 21 titik data mentah. Ia tidak tahu apa arti "menunjuk" atau "mengepal". Anda harus memberitahunya menggunakan logika Python.

*Lalu bagaimana caranya?* Simpel, dengan cara mengukur jarak atau membandingkan posisi antar titik.

#### **Contoh A: Gestur "Menunjuk" ☝️**

**Logika:** "Sebuah tangan dianggap 'menunjuk' JIKA..."
- Ujung jari telunjuk (Titik 8) lurus dan jauh dari telapak tangan
- **DAN** ujung jari tengah (Titik 12), jari manis (Titik 16), dan kelingking (Titik 20) posisinya "melipat" atau dekat dengan telapak tangan

#### **Contoh B: Gestur "Tangan Mengepal" ✊**

**Logika:** "Sebuah tangan dianggap 'mengepal' JIKA..."
- Semua ujung jari (Titik #8, #12, #16, #20) posisinya dekat dengan telapak tangan
- Jempol (Titik #4) juga dalam posisi melipat

#### Kesimpulan

Dengan membandingkan posisi relatif titik-titik landmark tangan, Anda dapat mengartikan gestur tangan tertentu. Logika ini dapat diperluas untuk mengenali gestur yang lebih kompleks sesuai kebutuhan aplikasi kita.

### Praktik: Menghitung jumlah jari yang terangkat

Untuk menghitung jumlah jari yang terangkat, kita bisa melakukan deteksi ujung dan dasar dari masing-masing jari kemudian membandingkan titiknya secara vertikal:

- jika ujung jari lebih tinggi dari dasar/tengah jarinya, maka jari tersebut terangkat. Secara programatik: `finger_tip.y < finger_base.y`

*Catatan: nilai y=0 dimulai dari atas frame. Artinya, nilai y yang lebih kecil menandakan titik ada di bagian atas dan nilai y yang lebih besar ada di bagian bawah*

In [9]:
import cv2
import numpy as np
import math

# 1. Buka webcam bawaan laptop
cap = cv2.VideoCapture(0)

print("Membuka kamera... Masukkan telapak tangan Anda ke dalam area kotak hijau.")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        print("Gagal mengambil frame dari kamera.")
        break
        
    # Efek cermin agar gerakan tangan natural di layar
    frame = cv2.flip(frame, 1)
    h, w, _ = frame.shape
    
    # 2. Tentukan Wilayah Deteksi (ROI) berupa kotak hijau di sisi kanan layar
    roi_x_start, roi_y_start = int(w * 0.5), int(h * 0.2)
    roi_x_end, roi_y_end = int(w * 0.95), int(h * 0.8)
    cv2.rectangle(frame, (roi_x_start, roi_y_start), (roi_x_end, roi_y_end), (0, 255, 0), 2)
    
    # Potong gambar agar sistem hanya memproses area di dalam kotak hijau
    roi = frame[roi_y_start:roi_y_end, roi_x_start:roi_x_end]
    
    # 3. Pra-pemrosesan Citra (Ubah ke HSV untuk segmentasi warna kulit)
    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    
    # Batas ambang warna kulit manusia pada ruang warna HSV
    lower_skin = np.array([0, 20, 70], dtype=np.uint8)
    upper_skin = np.array([20, 255, 255], dtype=np.uint8)
    
    # Membuat masker biner (hitam-putih) berdasarkan warna kulit
    mask = cv2.inRange(hsv, lower_skin, upper_skin)
    kernel = np.ones((5, 5), np.uint8)
    
    # PERBAIKAN: Menggunakan cv2.dilate (bukan cv2.dilation) untuk menghilangkan bintik hitam
    mask = cv2.dilate(mask, kernel, iterations=1)
    
    # PERBAIKAN OPTIMASI: Menggunakan sigmaX = 0 agar efek blur pas dan tidak merusak bentuk kontur
    mask = cv2.GaussianBlur(mask, (5, 5), 0)
    
    # 4. Cari Kontur Objek Terbesar (Tangan)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    finger_count = 0
    
    if len(contours) > 0:
        # Ambil objek dengan luas area terbesar di dalam kotak
        contour = max(contours, key=cv2.contourArea)
        
        # Jalankan logika jika luas objek cukup besar (memastikan itu adalah tangan)
        if cv2.contourArea(contour) > 5000:
            # Convex Hull (Mencari titik terluar/ujung telapak tangan)
            hull = cv2.convexHull(contour, returnPoints=False)
            
            # Convexity Defects (Mencari celah terdalam/lembah di antara jari)
            defects = cv2.convexityDefects(contour, hull)
            
            if defects is not None:
                for i in range(defects.shape[0]):
                    s, e, f, d = defects[i, 0]
                    start = tuple(contour[s][0]) # Titik ujung jari A
                    end = tuple(contour[e][0])   # Titik ujung jari B
                    far = tuple(contour[f][0])   # Titik celah terdalam
                    
                    # Hitung panjang sisi-sisi segitiga antar titik menggunakan koordinat
                    a = math.sqrt((end[0] - start[0])**2 + (end[1] - start[1])**2)
                    b = math.sqrt((far[0] - start[0])**2 + (far[1] - start[1])**2)
                    c = math.sqrt((end[0] - far[0])**2 + (end[1] - far[1])**2)
                    
                    # Rumus Hukum Kosinus untuk mencari sudut celah jari
                    angle = math.acos((b**2 + c**2 - a**2) / (2 * b * c)) * 57
                    
                    # Jari dihitung terangkat jika sudut celahnya lancip (<= 90 derajat)
                    # dan kedalaman celahnya (d) cukup signifikan
                    if angle <= 90 and d > 3000:
                        finger_count += 1
                        # Gambar titik merah di setiap celah jari yang valid
                        cv2.circle(roi, far, 5, [0, 0, 255], -1)
                
                # Rumus matematika: Jumlah jari terangkat = Jumlah celah + 1
                if finger_count > 0:
                    finger_count += 1
                else:
                    # Antisipasi jika hanya ada 1 jari tegak (0 celah terdeteksi)
                    finger_count = 1
            
            # Gambar garis tepi tangan (kontur) dengan warna biru
            cv2.drawContours(roi, [contour], -1, (255, 0, 0), 2)
            
    # 5. Tampilkan Informasi Teks ke Layar Utama
    cv2.putText(frame, f"Fingers: {finger_count}", (10, 70), 
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.putText(frame, "Masukkan tangan ke kotak hijau", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
    cv2.putText(frame, "Tekan 'q' untuk keluar", (10, h - 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    
    # Tampilkan jendela aplikasi
    cv2.imshow("Finger Counter (OpenCV Pure)", frame)
    
    # Keluar dari aplikasi jika menekan tombol 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Lepaskan resource kamera dan tutup jendela window
cap.release()
cv2.destroyAllWindows()
print("Aplikasi berhasil ditutup dengan aman.")

Membuka kamera... Masukkan telapak tangan Anda ke dalam area kotak hijau.
Aplikasi berhasil ditutup dengan aman.


#### 🔎 Eksplorasi

- Menggunakan logika sederhana yang sudah kita implementasikan, meskipun kita telah mengepalkan tangan kita, program masih membaca ada satu jari yang terangkat. Jari apakah yang menyebabkan hal tersebut? dan ide apa yang kamu punya untuk mengatasi masalah ini?

*Hint: Jempol memiliki orientasi berbeda, mungkin perlu logika terpisah*

### Praktik: Membedakan gestur kepalan tangan VS telapak terbuka

Untuk membedakan gestur kepalan tangan vs telapak terbuka, kita dapat menganalisis posisi landmark jari-jari relatif terhadap telapak tangan. Berikut adalah tahapan implementasinya:

1. Analisis Posisi Jari
Kita membandingkan posisi ujung jari (tip) dengan sendi tengah (PIP) atau pangkal jari:
- **Jari terangkat**: Ujung jari berada di atas sendi tengah (koordinat y lebih kecil)
- **Jari tertutup**: Ujung jari berada di bawah atau sejajar dengan sendi tengah

2. Logika Penentuan Gestur

**Kepalan Tangan (Fist):**
- Semua ujung jari (index 8, 12, 16, 20) berada di bawah sendi tengahnya
- Jempol (index 4) tertutup ke dalam telapak tangan
- Kondisi: `finger_count == 0` atau semua jari dalam posisi tertutup

**Telapak Terbuka (Open Palm):**
- Semua ujung jari berada di atas sendi tengahnya
- Jempol terangkat dan terpisah dari telapak tangan
- Kondisi: `finger_count >= 4` dan jarak antar jari cukup lebar, atau `tip.y < base.y`

In [19]:
import cv2
import numpy as np
import math

# 1. Buka webcam bawaan laptop
cap = cv2.VideoCapture(0)

print("Membuka kamera... Masukkan tangan Anda ke area kotak hijau.")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
        
    # Efek cermin agar gerakan tangan natural di layar
    frame = cv2.flip(frame, 1)
    h, w, _ = frame.shape
    
    # 2. Tentukan Wilayah Deteksi (ROI) kotak hijau di sisi kanan layar
    roi_x_start, roi_y_start = int(w * 0.5), int(h * 0.2)
    roi_x_end, roi_y_end = int(w * 0.95), int(h * 0.8)
    cv2.rectangle(frame, (roi_x_start, roi_y_start), (roi_x_end, roi_y_end), (0, 255, 0), 2)
    
    # Potong gambar khusus area dalam kotak hijau
    roi = frame[roi_y_start:roi_y_end, roi_x_start:roi_x_end]
    
    # 3. Segmentasi Warna Kulit dengan HSV
    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    lower_skin = np.array([0, 20, 70], dtype=np.uint8)
    upper_skin = np.array([20, 255, 255], dtype=np.uint8)
    
    mask = cv2.inRange(hsv, lower_skin, upper_skin)
    kernel = np.ones((5, 5), np.uint8)
    
    # Dilate dan blur untuk menghaluskan masker biner dari noise
    mask = cv2.dilate(mask, kernel, iterations=1)
    mask = cv2.GaussianBlur(mask, (5, 5), 0)
    
    # 4. Cari Kontur Tangan
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    finger_count = 0
    gesture_open_palm = False
    
    if len(contours) > 0:
        contour = max(contours, key=cv2.contourArea)
        
        if cv2.contourArea(contour) > 5000:
            # Analisis Convex Hull & Defects (Mencari sela-sela jari)
            hull = cv2.convexHull(contour, returnPoints=False)
            defects = cv2.convexityDefects(contour, hull)
            
            if defects is not None:
                for i in range(defects.shape[0]):
                    s, e, f, d = defects[i, 0]
                    start = tuple(contour[s][0])
                    end = tuple(contour[e][0])
                    far = tuple(contour[f][0])
                    
                    # Hitung jarak antar titik untuk mencari sudut celah jari
                    a = math.sqrt((end[0] - start[0])**2 + (end[1] - start[1])**2)
                    b = math.sqrt((far[0] - start[0])**2 + (far[1] - start[1])**2)
                    c = math.sqrt((end[0] - far[0])**2 + (end[1] - far[2])**2) if len(far) > 2 else math.sqrt((end[0] - far[0])**2 + (end[1] - far[1])**2)
                    
                    # Hukum Kosinus
                    angle = math.acos((b**2 + c**2 - a**2) / (2 * b * c)) * 57
                    
                    # Jika sudut celah jari terbuka tajam (<= 90 derajat)
                    if angle <= 90 and d > 3000:
                        finger_count += 1
                        cv2.circle(roi, far, 5, [255, 0, 0], -1) # Titik biru di sela jari
                
                # Rumus total jari terangkat = jumlah celah + 1
                if finger_count > 0:
                    finger_count += 1
            
            # Tentukan status gestur berdasarkan jumlah jari terangkat
            if finger_count >= 3:
                gesture_open_palm = True
            
            # Gambar garis kontur tangan luar (Warna Putih)
            cv2.drawContours(roi, [contour], -1, (255, 255, 255), 2)
            
    # 5. Output Visual Status Gestur ke Layar Utama
    if gesture_open_palm:
        status_text = "OPEN PALM"
        color = (0, 255, 0) # Hijau jika membuka
    else:
        status_text = "FIST"
        color = (0, 0, 255) # Merah jika mengepal
        
    cv2.putText(frame, f"Status: {status_text}", (20, 50), 
                cv2.FONT_HERSHEY_SIMPLEX, 1, color, 3)
    cv2.putText(frame, f"Jari Terdeteksi: {finger_count}", (20, 90), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
    cv2.putText(frame, "Tekan 'q' untuk keluar", (20, h - 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)
    
    cv2.imshow("Fist vs Open Palm (Pure OpenCV)", frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print("Program sukses ditutup.")

Membuka kamera... Masukkan tangan Anda ke area kotak hijau.
Program sukses ditutup.


## Proyek Mini: Virtual Painter

Kita akan membuat sebuah aplikasi Virtual Painter yang memungkinkan pengguna menggambar di layar menggunakan gestur tangan:

- Menggabungkan deteksi tangan dan gestur
- Gestur jari telunjuk terangkat -> menggambar
- Gestur telapak tangan terbuka -> mengubah warna
- Gestur tangan ditutup -> menghapus gambar

In [22]:
import cv2
import numpy as np

def hand_drawing_opencv():
    # 1. Inisialisasi Kamera Webcam
    cap = cv2.VideoCapture(0)
    
    if not cap.isOpened():
        print("Error: Kamera tidak dapat dibuka.")
        return
        
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    
    print(f"Video dimensions: {width}x{height}, FPS: {fps}")
    
    # 2. Parameter Menggambar
    drawing_color = (0, 0, 255)  # Warna coretan: Merah (Format BGR)
    drawing_thickness = 5
    
    # Membuat kanvas hitam kosong dengan resolusi yang sama dengan kamera
    canvas = np.zeros((height, width, 3), dtype=np.uint8)
    prev_pointer_pos = None

    # 3. Penentuan Range Warna Objek Penunjuk (HSV)
    # Range di bawah ini disetel untuk mendeteksi warna BIRU CERAH
    lower_color = np.array([90, 100, 100])
    upper_color = np.array([130, 255, 255])

    print("Aplikasi Menggambar Aktif! Silakan gerakkan penunjuk warna Biru ke kamera.")

    while True:
        ret, frame = cap.read()
        if not ret:
            print("Gagal mengambil gambar dari webcam.")
            break
            
        # Balik frame secara horizontal untuk efek cermin (mirror effect) yang natural
        frame = cv2.flip(frame, 1)
        
        # Konversi ruang warna dari BGR ke HSV
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        
        # Segmentasi objek berdasarkan range warna yang ditentukan
        mask = cv2.inRange(hsv, lower_color, upper_color)
        
        # Pembersihan noise masker biner menggunakan Erosi dan Dilasi
        mask = cv2.erode(mask, None, iterations=2)
        
        # SOLUSI TEPAT: Menggunakan cv2.dilate (bukan cv2.dilation)
        mask = cv2.dilate(mask, None, iterations=2)
        
        # Menghaluskan tepi masker biner dengan Gaussian Blur (SigmaX auto-compute = 0)
        mask = cv2.GaussianBlur(mask, (5, 5), 0)
        
        # 4. Temukan Kontur Objek Penunjuk
        contours, _ = cv2.findContours(mask.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        center = None
        
        # Jika ada objek dengan warna yang sesuai terdeteksi
        if len(contours) > 0:
            # Cari kontur dengan area permukaan terbesar
            c = max(contours, key=cv2.contourArea)
            
            # Buat lingkaran pembungkus terkecil di sekitar kontur objek tersebut
            ((x, y), radius) = cv2.minEnclosingCircle(c)
            
            # Proses pelacakan hanya berjalan jika ukuran objek cukup signifikan (bukan noise)
            if radius > 10:
                center = (int(x), int(y))
                
                # Gambar lingkaran tracker warna hijau pada frame asli sebagai penanda kuas
                cv2.circle(frame, center, int(radius), (0, 255, 0), 2)
                cv2.circle(frame, center, 5, (0, 0, 255), -1)
                
                # Gambar garis kontinu pada kanvas jika posisi pointer sebelumnya terekam
                if prev_pointer_pos is not None:
                    cv2.line(canvas, prev_pointer_pos, center, drawing_color, drawing_thickness)
                
                # Perbarui posisi koordinat pointer terakhir
                prev_pointer_pos = center
        else:
            # Putus koordinat jika objek keluar dari kamera agar garis tidak melompat jauh
            prev_pointer_pos = None
            
        # 5. Gabungkan Frame Kamera Utama dengan Kanvas Gambar (Transparansi Masing-masing 70%)
        combined_image = cv2.addWeighted(frame, 0.7, canvas, 0.7, 0)
        
        # Tampilkan teks informasi panduan navigasi di layar
        cv2.putText(combined_image, "Gerakkan objek biru untuk menggambar", (10, 30), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        cv2.putText(combined_image, "Tekan 'c' untuk membersihkan kanvas", (10, 60), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
        cv2.putText(combined_image, "Tekan 'q' untuk keluar aplikasi", (10, 90), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
        
        # Tampilkan window visualisasi aplikasi
        cv2.imshow('Virtual Hand Drawing App', combined_image)
        
        # Pengendalian tombol keyboard
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            break
        elif key == ord('c'):
            # Bersihkan kanvas dengan mengisinya kembali menggunakan array nol (hitam)
            canvas = np.zeros((height, width, 3), dtype=np.uint8)
            prev_pointer_pos = None
            print("Kanvas berhasil dibersihkan.")
            
    # Lepaskan resource webcam dan hancurkan semua jendela tampilan
    cap.release()
    cv2.destroyAllWindows()
    print("Aplikasi ditutup dengan aman.")

# Jalankan Fungsi Utama
hand_drawing_opencv()

Video dimensions: 640x480, FPS: 30
Aplikasi Menggambar Aktif! Silakan gerakkan penunjuk warna Biru ke kamera.
Aplikasi ditutup dengan aman.


Dari proyek mini ini, ada beberapa topik penting dalam mencapai fungsionalitas yang kita inginkan:

### Menggabungkan deteksi tangan dan gestur

Tahapan:
- Landmark tangan di frame saat ini dideteksi menggunakan mediapipe dengan sintaks `results = hands.process(rgb_frame)`
- Dapatkan posisi dari ujung dan dasar jari telunjuk
- Jika ujung jari lebih tinggi dari dasar jari, maka buat garis dari titik sebelumnya: `cv2.line(canvas, prev_finger_pos, (x, y), drawing_color, drawing_thickness)`
- Simpan posisi jari sekarang sebagai referensi untuk frame berikutnya
- sebaliknya, Jika ujung jari lebih rendah dari dasar jari, hentikan penggambaran garis dan skip frame saat ini `else: prev_finger_pos = None`


### Menggunakan gestur (misal: jari telunjuk terangkat) untuk menggambar di layar.

- Jika ujung jari lebih tinggi dari dasar jari, maka buat garis dari titik sebelumnya: `cv2.line(canvas, prev_finger_pos, (x, y), drawing_color, drawing_thickness)`
- Simpan posisi jari sekarang sebagai referensi untuk frame berikutnya
- sebaliknya, Jika ujung jari lebih rendah dari dasar jari, hentikan penggambaran garis dan skip frame saat ini `else: prev_finger_pos = None`

### Menggunakan gestur lain (telapak terbuka) untuk menghapus kanvas.

Tahapan deteksi gestur telapak terbuka:
- Periksa status semua jari (telunjuk, tengah, manis, kelingking) dengan membandingkan posisi y ujung vs pangkal jari
- Jari dianggap "terangkat" jika koordinat y ujung lebih kecil dari y pangkal: `index_up = y < base_y`
- Untuk jari lainnya: `middle_up = hand_landmarks.landmark[12].y < hand_landmarks.landmark[9].y`
- Jika semua jari terangkat (`is_open_palm = index_up and middle_up and ring_up and pinky_up`), hapus kanvas
- Reset kanvas dengan membuat array kosong: `canvas = np.zeros((height, width, 3), dtype=np.uint8)`
- Reset posisi jari sebelumnya: `prev_finger_pos = None`

## Kesimpulan

Dalam modul **Video Processing Week 3: Analisis Tangan & Aplikasi Interaktif**, kita telah mempelajari implementasi teknologi AI untuk deteksi dan pengenalan gestur tangan menggunakan MediaPipe. Berikut adalah rangkuman pencapaian pembelajaran:

### 🎯 Pencapaian Utama

**1. Hand Landmark Detection**
- Berhasil mengimplementasikan deteksi 21 titik landmark pada tangan secara real-time
- Memahami struktur data landmark MediaPipe dan cara visualisasinya
- Mengonfigurasi parameter deteksi untuk optimalisasi akurasi dan performa

**2. Hand Gesture Recognition**
- Mengembangkan logika penerjemahan data landmark mentah menjadi informasi gestur
- Implementasi penghitung jari dengan membandingkan posisi relatif ujung dan pangkal jari
- Membedakan gestur kepalan tangan vs telapak terbuka menggunakan analisis posisi landmark

**3. Virtual Painter Application**
- Membangun aplikasi interaktif yang merespons gestur tangan secara real-time
- Mengintegrasikan multiple gesture recognition: menggambar (jari telunjuk), menghapus (telapak terbuka)
- Menerapkan teknik overlay canvas untuk visualisasi hasil gambar

### 🔑 Konsep Kunci yang Dipelajari

- **Coordinate System**: Memahami sistem koordinat MediaPipe (nilai y=0 di atas frame)
- **Landmark Analysis**: Teknik membandingkan posisi relatif antar landmark untuk interpretasi gestur
- **Real-time Processing**: Implementasi pipeline deteksi dan response dalam aplikasi interaktif
- **Computer Vision Integration**: Menggabungkan OpenCV dan MediaPipe untuk solusi vision yang kompleks

### 💡 Aplikasi Praktis

Materi ini memberikan foundation yang kuat untuk pengembangan:
- Aplikasi kontrol gestur untuk presentasi atau gaming
- Sistem antarmuka touchless untuk lingkungan steril
- Aplikasi edukasi interaktif untuk anak-anak
- Prototyping human-computer interaction yang inovatif

### 🚀 Pengembangan Selanjutnya

Dengan pemahaman dasar ini, pembelajaran dapat dilanjutkan ke:
- Gesture recognition yang lebih kompleks (sign language, custom gestures)
- Integration dengan machine learning untuk pattern recognition
- Multi-modal interaction (kombinasi hand tracking dengan voice/eye tracking)
- Optimalisasi performa untuk deployment pada edge devices